In [1]:
import matplotlib.pyplot as plt
import numpy as np

import neuron
from neuron import h
from hnn_core.params import compare_dictionaries

neuron.load_mechanisms('/Users/katharinaduecker/Documents/projects_brown/hnn-tuning/local_hnn/hnn_core/mod')


# import os
# os.mkdir(figpth)# 
from cell_builder_utils import l5_params
import hnn_core
from hnn_core import Cell
from hnn_core.cells_default import (_get_dends, _get_pyr_soma, _get_pyr_syn_props, _get_mechanisms, _exp_g_at_dist, _linear_g_at_dist,
                                    _get_basket_soma, _get_basket_syn_props)

from functools import partial

from hnn_core.viz import plot_cell_morphology
from hnn_core.params_default import (get_L2Pyr_params_new, get_L5Pyr_params_new, get_Int_params)

# import pandas as pd
from scipy import signal
from plot_utils import fig_conductances, fig_compartments
from matplotlib.gridspec import GridSpec
import os
import matplotlib as mpl
mpl.rcParams['pdf.fonttype'] = 42
mpl.rcParams['ps.fonttype'] = 42

In [2]:
# build L5 cell
pos = (0,0,0)
gid = None

p_all = get_L5Pyr_params_new()


# override params according to function
gbar_Ca_HVA = partial(_linear_g_at_dist, gsoma=p_all['L5Pyr_dend_gbar_Ca_HVA'], gdend=p_all['L5Pyr_dend_gbar_Ca_HVA'] *9, xkink=1500, hotzone_factor=3)
gbar_Ca_LVA = partial(_linear_g_at_dist, gsoma=p_all['L5Pyr_dend_gbar_Ca_LVAst'], gdend=p_all['L5Pyr_dend_gbar_Ca_LVAst']*9, xkink=1500, hotzone_factor=3)
gbar_Ih = partial(_exp_g_at_dist, zero_val=p_all['L5Pyr_dend_gbar_Ih'],exp_term = 1.1/323, slope=5, offset=-.8696)

override_params = dict()
override_params['L5Pyr_dend_gbar_Ca_HVA'] = gbar_Ca_HVA
override_params['L5Pyr_dend_gbar_Ca_LVAst'] = gbar_Ca_LVA
override_params['L5Pyr_dend_gbar_Ih'] = gbar_Ih

p_all = compare_dictionaries(p_all, override_params)

end_pts = {
        'soma': [[0, 0, 0], [0, 0, 23]],
        'apical_trunk': [[0, 0, 23], [0, 0, 83]],
        'apical_oblique': [[0, 0, 83], [-150, 0, 83]],
        'apical_1': [[0, 0, 83], [0, 0, 483]],
        'apical_2': [[0, 0, 483], [0, 0, 883]],
        'apical_tuft': [[0, 0, 883], [0, 0, 1133]],
        'basal_1': [[0, 0, 0], [0, 0, -50]],
        'basal_2': [[0, 0, -50], [-106, 0, -156]],
        'basal_3': [[0, 0, -50], [106, 0, -156]]
    }

cell_tree = {
        ('apical_trunk', 0): [('apical_trunk', 1)],
        ('apical_1', 0): [('apical_1', 1)],
        ('apical_2', 0): [('apical_2', 1)],
        ('apical_tuft', 0): [('apical_tuft', 1)],
        ('apical_oblique', 0): [('apical_oblique', 1)],
        ('basal_1', 0): [('basal_1', 1)],
        ('basal_2', 0): [('basal_2', 1)],
        ('basal_3', 0): [('basal_3', 1)],
        # Different sections connected
        ('soma', 0): [('soma', 1), ('basal_1', 0)],
        ('soma', 1): [('apical_trunk', 0)],
        ('apical_trunk', 1): [('apical_1', 0), ('apical_oblique', 0)],
        ('apical_1', 1): [('apical_2', 0)],
        ('apical_2', 1): [('apical_tuft', 0)],
        ('basal_1', 1): [('basal_2', 0), ('basal_3', 0)]
        }

# build sections
section_names = list(end_pts.keys())

sections = _get_dends(p_all, 'L5Pyr', section_names)
sections['soma'] = _get_pyr_soma(p_all, 'L5Pyr')

# Soma and apical mechanisms
mechanisms = {'NaTs2_t': ['gbar_NaTs2_t'], 
                'SKv3_1': ['gbar_SKv3_1'],
                'Nap_Et2': ['gbar_Nap_Et2'],
                'Ca_HVA': ['gbar_Ca_HVA'],
                'Ca_LVAst': ['gbar_Ca_LVAst'],
                'SK_E2': ['gbar_SK_E2'],
                'pas': ['g_pas', 'e_pas'],
                'Ih': ['gbar_Ih'],
                'Im': ['gbar_Im'],
                'K_Pst': ['gbar_K_Pst'],
                'K_Tst': ['gbar_K_Tst'],
                'NaTa_t':['gbar_NaTa_t'],
                'CaDynamics_E2': ['decay_CaDynamics_E2', 'gamma_CaDynamics_E2']}

p_mech_soma = _get_mechanisms(p_all, 'L5Pyr', ['soma'], mechanisms)

section_names = ['apical_trunk', 'apical_1', 'apical_2', 'apical_tuft']

p_mech_apical = _get_mechanisms(p_all, 'L5Pyr', section_names, mechanisms)


# basal mechanisms
mechanisms = {'NaTs2_t': ['gbar_NaTs2_t'], 
                'SKv3_1': ['gbar_SKv3_1'],
                'pas': ['g_pas', 'e_pas'],
                'Ih': ['gbar_Ih']}

section_names = ['basal_1', 'basal_2', 'basal_3', 'apical_oblique']
p_mech_basal = _get_mechanisms(p_all, 'L5Pyr', section_names, mechanisms)


p_mech = {**p_mech_soma, **p_mech_basal, **p_mech_apical}


for sec_name, section in sections.items():
        section._end_pts = end_pts[sec_name]

        if sec_name == 'soma':
            section.syns = ['gabaa', 'gabab']
        else:
            section.syns = ['ampa', 'nmda', 'gabaa', 'gabab']

        section.mechs = p_mech[sec_name]

sect_loc = {'proximal': ['apical_oblique', 'basal_2', 'basal_3'],
                'distal': ['apical_tuft']}

synapses = _get_pyr_syn_props(p_all, 'L5Pyr')



cell = Cell('L5Pyr', pos,
                sections=sections,
                synapses=synapses,
                sect_loc=sect_loc,
                cell_tree=cell_tree,
                gid=gid)

cell.build()



NaTs2_t
SKv3_1
Nap_Et2
Ca_HVA
Ca_LVAst
SK_E2
pas
Ih
Im
K_Pst
K_Tst
NaTa_t
CaDynamics_E2
NaTs2_t
SKv3_1
Nap_Et2
Ca_HVA
Ca_LVAst
SK_E2
pas
Ih
Im
K_Pst
K_Tst
NaTa_t
CaDynamics_E2
NaTs2_t
SKv3_1
pas
Ih
NaTs2_t
SKv3_1
Nap_Et2
Ca_HVA
Ca_LVAst
SK_E2
pas
Ih
Im
K_Pst
K_Tst
NaTa_t
CaDynamics_E2
NaTs2_t
SKv3_1
Nap_Et2
Ca_HVA
Ca_LVAst
SK_E2
pas
Ih
Im
K_Pst
K_Tst
NaTa_t
CaDynamics_E2
NaTs2_t
SKv3_1
Nap_Et2
Ca_HVA
Ca_LVAst
SK_E2
pas
Ih
Im
K_Pst
K_Tst
NaTa_t
CaDynamics_E2
NaTs2_t
SKv3_1
pas
Ih
NaTs2_t
SKv3_1
pas
Ih
NaTs2_t
SKv3_1
pas
Ih


In [3]:
# from hnn_core.cells_default import pyramidal_l5

# l5_cell = pyramidal_l5((0,0,0),gid=1)

p_all = get_L2Pyr_params_new()

p_all['L2Pyr_soma_gbar_Ih']

8e-05

In [4]:
# build L2/3 cell
p_all = get_L2Pyr_params_new()

gbar_Ih = partial(_exp_g_at_dist, zero_val=p_all['L2Pyr_dend_gbar_Ih'],exp_term = 1/110, slope=2.087+5, offset=-0.869600)

override_params = dict()
override_params['L2Pyr_dend_gbar_Ih'] = gbar_Ih

p_all = compare_dictionaries(p_all, override_params)

end_pts = {
        'soma': [[-50, 0, 765], [-50, 0, 778]],
        'apical_trunk': [[-50, 0, 778], [-50, 0, 813]],
        'apical_oblique': [[-50, 0, 813], [-250, 0, 813]],
        'apical_1': [[-50, 0, 813], [-50, 0, 993]],
        'apical_tuft': [[-50, 0, 993], [-50, 0, 1133]],
        'basal_1': [[-50, 0, 765], [-50, 0, 715]],
        'basal_2': [[-50, 0, 715], [-156, 0, 609]],
        'basal_3': [[-50, 0, 715], [56, 0, 609]],
    }

cell_tree = {
('apical_trunk', 0): [('apical_trunk', 1)],
('apical_1', 0): [('apical_1', 1)],
('apical_tuft', 0): [('apical_tuft', 1)],
('apical_oblique', 0): [('apical_oblique', 1)],
('basal_1', 0): [('basal_1', 1)],
('basal_2', 0): [('basal_2', 1)],
('basal_3', 0): [('basal_3', 1)],
# Different sections connected
('soma', 0): [('soma', 1), ('basal_1', 0)],
('soma', 1): [('apical_trunk', 0)],
('apical_trunk', 1): [('apical_1', 0), ('apical_oblique', 0)],
('apical_1', 1): [('apical_tuft', 0)],
('basal_1', 1): [('basal_2', 0), ('basal_3', 0)]}

# build sections
section_names = list(end_pts.keys())

sections = _get_dends(p_all, 'L2Pyr', section_names)
sections['soma'] = _get_pyr_soma(p_all, 'L2Pyr')


mechanisms = {'NaTs2_t': ['gbar_NaTs2_t'], 
                'SKv3_1': ['gbar_SKv3_1'],
                'Nap_Et2': ['gbar_Nap_Et2'],
                'Ca_HVA': ['gbar_Ca_HVA'],
                'Ca_LVAst': ['gbar_Ca_LVAst'],
                'SK_E2': ['gbar_SK_E2'],
                'pas': ['g_pas', 'e_pas'],
                'Ih': ['gbar_Ih'],
                'Im': ['gbar_Im'],
                'CaDynamics_E2': ['decay_CaDynamics_E2', 'gamma_CaDynamics_E2']}


p_mech_soma = _get_mechanisms(p_all, 'L2Pyr', ['soma'], mechanisms)

# apical sections

mechanisms = {'NaTa_t': ['gbar_NaTa_t'], 
                'SKv3_1': ['gbar_SKv3_1'],
                'Nap_Et2': ['gbar_Nap_Et2'],
                'Ca_HVA': ['gbar_Ca_HVA'],
                'Ca_LVAst': ['gbar_Ca_LVAst'],
                'SK_E2': ['gbar_SK_E2'],
                'pas': ['g_pas', 'e_pas'],
                'Ih': ['gbar_Ih'],
                'Im': ['gbar_Im']}

section_names = ['apical_trunk', 'apical_1', 'apical_tuft']
p_mech_apical = _get_mechanisms(p_all, 'L2Pyr', section_names, mechanisms)

# basal sections
mechanisms = {'NaTa_t': ['gbar_NaTa_t'], 
                'SKv3_1': ['gbar_SKv3_1'],
                'pas': ['g_pas', 'e_pas'],
                'Ih': ['gbar_Ih']}

section_names = ['basal_1', 'basal_2', 'basal_3', 'apical_oblique']

p_mech_basal = _get_mechanisms(p_all, 'L2Pyr', section_names, mechanisms)


p_mech = {**p_mech_soma, **p_mech_basal, **p_mech_apical}

for sec_name, section in sections.items():
        section._end_pts = end_pts[sec_name]

        if sec_name == 'soma':
            section.syns = ['gabaa', 'gabab']
        else:
            section.syns = ['ampa', 'nmda', 'gabaa', 'gabab']

        section.mechs = p_mech[sec_name]


cell = Cell('L2Pyr', pos,
                sections=sections,
                synapses=synapses,
                sect_loc=sect_loc,
                cell_tree=cell_tree,
                gid=gid)

cell.build()

NaTs2_t
SKv3_1
Nap_Et2
Ca_HVA
Ca_LVAst
SK_E2
pas
Ih
Im
CaDynamics_E2
NaTa_t
SKv3_1
Nap_Et2
Ca_HVA
Ca_LVAst
SK_E2
pas
Ih
Im
NaTa_t
SKv3_1
pas
Ih
NaTa_t
SKv3_1
Nap_Et2
Ca_HVA
Ca_LVAst
SK_E2
pas
Ih
Im
NaTa_t
SKv3_1
Nap_Et2
Ca_HVA
Ca_LVAst
SK_E2
pas
Ih
Im
NaTa_t
SKv3_1
pas
Ih
NaTa_t
SKv3_1
pas
Ih
NaTa_t
SKv3_1
pas
Ih


In [12]:
cell_name = 'L2Basket'
gid = 0
pos = (0,0,0)
p_all = get_Int_params()
sections = dict()
sections['soma'] = _get_basket_soma(cell_name)
synapses = _get_basket_syn_props()
sections['soma'].syns = list(synapses.keys())

if cell_name == 'L2Basket':
    sect_loc = dict(proximal=['soma'], distal=['soma'])
elif cell_name == 'L5Basket':
    sect_loc = dict(proximal=['soma'], distal=[])

cell_tree = None

mechanisms = {'nas': ['gbar_nas'],
              'kdr': ['gbar_kdr'],
              'kd': ['gbar_kd'],
              'Ih': ['gbar_Ih'],
              'pas': ['g_pas', 'e_pas']}

sections['soma'].mechs = dict()

for mech_name in mechanisms:
    for mech_attr in mechanisms[mech_name]:
        this_mech_prop = dict()
        key = f'Int_{mech_attr}'
        this_mech_prop[mech_attr] = p_all[key]
        sections['soma'].mechs[mech_name] = this_mech_prop


cell = Cell(cell_name, pos,
            sections=sections,
            synapses=synapses,
            sect_loc=sect_loc,
            cell_tree=cell_tree,
            gid=gid)


In [ ]:



cell.build()

mechs = params['soma'].keys()
for i,m in enumerate(mechs):
    cell._nrn_sections['soma'].insert(m)
    for seg in cell._nrn_sections['soma']:
        if m == 'pas':
            seg.pas.g = params['soma']['pas']['g']
            seg.pas.e = params['soma']['pas']['e']
        else:
            setattr(seg, 'gbar_'+m, params['soma'][m])